In [33]:
import ollama
import langchain
from langchain_ollama import ChatOllama
import os
from IPython.display import Markdown, display

In [24]:
os.environ["NO_PROXY"]="localhost,127.0.0.1,::1,localhost:22366,127.0.0.1,22366"
os.environ["no_proxy"]="localhost,127.0.0.1,::1,localhost:22366,127.0.0.1,22366"


In [25]:
llm = ChatOllama(
    model="gemma3:1b",
    base_url="http://127.0.0.1:22366",
    temperature=0)
response = llm.invoke("Explain LangChain in one sentence.")
print(response.content)

LangChain is a framework for building applications powered by large language models (LLMs), simplifying the process of connecting LLM capabilities to other data sources and tools.


In [30]:
llm=ChatOllama(model="gemma4:31b",
               base_url="http://127.0.0.1:22366",
               temperature=0)

In [36]:
data=llm.invoke("Give me a pipeline for analysying visium hd.  Only provide script for me so that I can copy paste.")

In [37]:
display(Markdown(data.content))

Since Visium HD produces massive datasets (bin sizes from 2µm to 16µm), this pipeline uses **Scanpy** and **Squidpy**. 

**Prerequisites:**
`pip install scanpy squidpy pandas matplotlib seaborn`

```python
import scanpy as sc
import squidpy as sq
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. LOAD DATA
# ==========================================
# Path to the folder containing 'filtered_feature_bc_matrix.h5' and 'spatial/'
# For Visium HD, choose your preferred bin size folder (e.g., 8um or 16um)
data_path = "path/to/visium_hd_bin8/" 

adata = sc.read_visium(data_path)
adata.var_names_make_unique()

# ==========================================
# 2. QUALITY CONTROL (QC)
# ==========================================
# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, inplace=True)

# Filter spots with very low counts or genes
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=10)

# Visualize QC
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'], jitter=0.4, multi_panel=True)

# ==========================================
# 3. NORMALIZATION & FEATURE SELECTION
# ==========================================
# Save raw counts for later (Differential Expression)
adata.raw = adata

# Log-normalization
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Identify highly variable genes
sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=2000)
adata = adata[:, adata.var.highly_variable].copy()

# Scaling
sc.pp.scale(adata, max_value=10)

# ==========================================
# 4. DIMENSIONALITY REDUCTION & CLUSTERING
# ==========================================
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=15)

# UMAP for visualization
sc.tl.umap(adata)

# Leiden Clustering
sc.tl.leiden(adata, resolution=0.5)

# ==========================================
# 5. SPATIAL VISUALIZATION
# ==========================================
# Plot UMAP and Spatial Clusters side-by-side
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sc.pl.umap(adata, color='leiden', show=False, ax=ax[0])
sc.pl.spatial(adata, color='leiden', spot_size=10, show=False, ax=ax[1])
plt.tight_layout()
plt.show()

# ==========================================
# 6. DIFFERENTIAL EXPRESSION (MARKERS)
# ==========================================
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')

# Plot top 5 genes per cluster on the spatial map
top_genes = sc.get.rank_genes_groups_df(adata, group=None).groupby('group').head(1).index.tolist() # Simplified
# Or manually specify: top_genes = ['GeneA', 'GeneB', 'GeneC']

sc.pl.spatial(adata, color=top_genes[:4], spot_size=10, cmap='viridis')

# ==========================================
# 7. SPATIAL ANALYSIS (SQUIDPY)
# ==========================================
# Calculate spatial neighbors based on coordinates
sq.gr.spatial_neighbors(adata)

# Compute spatial autocorrelation (Moran's I) to find spatially variable genes
sq.gr.spatial_autocorr(adata, cs=True)

# Plot the most spatially variable genes
sc.pl.spatial(adata, color='spatial_autocorr', spot_size=10)

# ==========================================
# 8. SAVE RESULTS
# ==========================================
adata.write("visium_hd_processed.h5ad")
```

### Key Notes for Visium HD:
1.  **Binning:** Visium HD data is provided in different bin sizes (2µm, 8µm, 16µm). If your RAM is limited, start with the **16µm** or **8µm** folders. The 2µm data is often too large for standard workstations.
2.  **`spot_size`:** In `sc.pl.spatial`, you must adjust `spot_size`. For HD, the default is usually too large; set it to a small value (e.g., `10` or smaller) depending on your bin size.
3.  **Memory:** If you encounter memory errors, use `adata = adata.to_dask()` or increase your swap space.